# Modelo completo de PLN — Classificação de intenções no atendimento do Metrô

Este notebook implementa um pipeline de **Processamento de Linguagem Natural (PLN)** de ponta a ponta para classificar mensagens de passageiros. Ele foi preparado para execução no **Google Colab** e não depende do repositório do projeto.

## O que será desenvolvido

1. carregamento de uma base didática ou upload de CSV próprio;
2. análise exploratória e pré-processamento de português;
3. separação estratificada em treino, validação e teste;
4. treinamento de **Word2Vec** somente com os dados de treino;
5. baseline com média dos vetores Word2Vec + Regressão Logística;
6. rede neural **BiLSTM** com camada de **Embedding** inicializada pelo Word2Vec;
7. avaliação com acurácia, F1 macro, relatório e matriz de confusão;
8. função de inferência com limiar de confiança;
9. exportação do modelo, vocabulário, encoder e Word2Vec.

> **Nota acadêmica:** a base embutida é pequena e sintética, adequada para demonstrar o pipeline. Para uso real, substitua-a por mensagens anonimizadas e rotuladas por especialistas.

In [ ]:
# Dependências adicionais do Colab (execute esta célula uma vez)
%pip install -q gensim unidecode joblib


In [ ]:
import json
import os
import random
import re
import shutil
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import (Bidirectional, Dense, Dropout, Embedding,
                                     GlobalMaxPooling1D, LSTM, SpatialDropout1D)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from unidecode import unidecode

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print('TensorFlow:', tf.__version__)
print('GPU disponível:', bool(tf.config.list_physical_devices('GPU')))


## 1. Configuração e dados

Por padrão, o notebook usa uma base didática balanceada. Para usar dados próprios, altere `USAR_CSV_PROPRIO = True` e envie um CSV UTF-8 com as colunas:

- `texto`: mensagem do passageiro;
- `intencao`: classe correta da mensagem.

Cada classe deve ter, idealmente, dezenas ou centenas de exemplos. O código interrompe a execução se alguma classe tiver menos de 5 registros.

In [ ]:
USAR_CSV_PROPRIO = False
NOME_CSV = 'dados_intencoes.csv'

dados_demo = {
    'acessibilidade': [
        'preciso de ajuda para embarcar com cadeira de rodas',
        'a estacao possui elevador acessivel',
        'como solicito apoio para uma pessoa com mobilidade reduzida',
        'sou deficiente visual e preciso de orientacao na plataforma',
        'tem rota acessivel ate a linha azul',
        'o elevador para cadeirante esta funcionando',
        'viajo com uma pessoa idosa que nao consegue usar escadas',
        'existe piso tatil nessa estacao',
        'posso receber auxilio de um funcionario no desembarque',
        'qual entrada tem acesso para cadeira de rodas',
        'preciso usar o metro com muletas',
        'ha atendimento para passageiro com baixa visao',
        'como encontro a plataforma sem usar escada',
        'meu acompanhante precisa de uma rota acessivel',
        'a estacao aceita cao guia',
        'quero avisar que precisarei de apoio no destino'
    ],
    'achados_perdidos': [
        'esqueci minha mochila dentro do trem',
        'onde fica o setor de achados e perdidos',
        'perdi meu celular na estacao',
        'encontrei uma carteira na plataforma',
        'como registro um objeto perdido',
        'deixei uma sacola no ultimo vagao',
        'meu documento caiu no metro',
        'achei um guarda chuva no banco',
        'quero consultar o protocolo do item perdido',
        'perdi a chave de casa durante a viagem',
        'alguem encontrou um notebook ontem',
        'como devolvo um objeto que encontrei',
        'minha mala ficou no trem',
        'nao encontro meus fones depois do desembarque',
        'qual telefone de objetos perdidos',
        'posso buscar um pertence encontrado em outra estacao'
    ],
    'bilhetes_tarifas': [
        'quanto custa a passagem do metro',
        'meu bilhete digital nao passou na catraca',
        'onde posso comprar um bilhete',
        'como recarrego o cartao de transporte',
        'a catraca nao leu o codigo no celular',
        'existe desconto para estudante',
        'posso pagar a tarifa com cartao',
        'meu saldo nao apareceu depois da recarga',
        'qual e o valor da integracao',
        'o bilhete unico funciona nesta linha',
        'como pedir reembolso de uma cobranca duplicada',
        'crianca paga passagem',
        'aceita pagamento por aproximacao',
        'onde consulto o saldo do meu cartao',
        'o aplicativo nao mostra meu bilhete',
        'posso usar o mesmo cartao duas vezes seguidas'
    ],
    'bicicletas': [
        'posso levar bicicleta no metro',
        'qual horario permite embarcar com bike',
        'bicicleta pode entrar durante a semana',
        'em qual vagao devo colocar minha bicicleta',
        'domingo posso viajar com bike',
        'ha limite de bicicletas por trem',
        'patinete pode ser transportado no metro',
        'preciso desmontar a bicicleta para embarcar',
        'posso levar bike eletrica',
        'qual carro e indicado para ciclistas',
        'feriado aceita bicicleta o dia inteiro',
        'tem bicicletario nesta estacao',
        'bicicleta dobravel pode entrar em qualquer horario',
        'como embarcar com uma bike sem atrapalhar',
        'o ultimo vagao aceita bicicletas',
        'quais sao as regras para transportar bicicleta'
    ],
    'reclamacoes_sugestoes': [
        'quero fazer uma reclamacao sobre o atendimento',
        'onde envio uma sugestao para o metro',
        'preciso registrar uma denuncia',
        'como acompanho o protocolo da reclamacao',
        'um funcionario me atendeu mal',
        'quero elogiar a equipe da estacao',
        'o trem estava muito sujo e quero reclamar',
        'qual canal da ouvidoria',
        'gostaria de sugerir uma melhoria',
        'como relato um problema ocorrido ontem',
        'quero registrar uma manifestacao',
        'nao recebi resposta do meu protocolo',
        'onde faco uma queixa formal',
        'posso avaliar o atendimento recebido',
        'preciso corrigir os dados de uma reclamacao',
        'qual o prazo de resposta da ouvidoria'
    ],
    'equipamentos': [
        'a escada rolante esta quebrada',
        'o elevador da estacao parou de funcionar',
        'como aviso sobre uma catraca com defeito',
        'tem uma maquina de recarga fora de servico',
        'a porta automatica nao abre',
        'qual alternativa para a escada interditada',
        'o painel de informacoes esta apagado',
        'preciso comunicar um equipamento quebrado',
        'o bebedouro da plataforma nao funciona',
        'quando o elevador volta a operar',
        'a esteira rolante esta em manutencao',
        'uma catraca ficou travada',
        'o interfone de emergencia esta com defeito',
        'nao consigo usar a maquina de bilhetes',
        'ha previsao de conserto da escada rolante',
        'o portao de acesso esta bloqueado'
    ],
    'status_operacao': [
        'a linha azul esta funcionando agora',
        'os trens estao com atraso',
        'qual o status da operacao neste momento',
        'a linha vermelha esta parada',
        'tem alguma falha na circulacao',
        'o intervalo entre os trens aumentou',
        'a operacao voltou ao normal',
        'existe lentidao na linha hoje',
        'por que o trem esta parado no tunel',
        'ha estacoes fechadas agora',
        'como esta a circulacao na linha verde',
        'tem previsao para normalizar a operacao',
        'estao ocorrendo testes na linha',
        'o servico esta interrompido',
        'quais linhas apresentam problemas',
        'a viagem esta demorando mais hoje'
    ],
    'horarios': [
        'que horas o metro abre',
        'qual e o horario do ultimo trem',
        'o metro funciona de madrugada',
        'ate que horas a estacao fica aberta',
        'qual horario de funcionamento aos domingos',
        'o servico abre mais cedo no feriado',
        'que horas encerra a operacao hoje',
        'a estacao funciona vinte e quatro horas',
        'qual o primeiro trem da manha',
        'o horario muda no fim de semana',
        'consigo embarcar depois da meia noite',
        'quando os portoes da estacao abrem',
        'qual horario da linha no sabado',
        'ate quando posso fazer baldeacao',
        'o ultimo embarque acontece que horas',
        'existe operacao especial durante a noite'
    ],
    'rotas_estacoes': [
        'como chegar na estacao republica',
        'qual linha vai para a se',
        'onde faco baldeacao para a linha amarela',
        'qual a melhor rota ate a paulista',
        'esta estacao tem saida para qual rua',
        'quantas paradas faltam para o destino',
        'preciso trocar de trem para chegar ao centro',
        'qual estacao fica perto do museu',
        'como vou da linha azul para a vermelha',
        'onde fica a plataforma sentido bairro',
        'qual acesso devo usar para a avenida',
        'essa linha passa pela estacao luz',
        'como planejo uma viagem com duas conexoes',
        'qual sentido devo pegar para o terminal',
        'a transferencia entre linhas e gratuita',
        'quero saber o caminho entre duas estacoes'
    ],
    'seguranca_emergencia': [
        'tem uma pessoa passando mal no vagao',
        'como aciono a seguranca do metro',
        'presenciei uma situacao de assedio',
        'ha um objeto suspeito na plataforma',
        'preciso de atendimento medico urgente',
        'onde fica o botao de emergencia',
        'uma crianca se perdeu da familia',
        'como denuncio um furto no trem',
        'tem fumaca dentro da estacao',
        'um passageiro caiu na plataforma',
        'quero chamar um agente imediatamente',
        'a porta fechou prendendo uma mochila',
        'alguem esta brigando no vagao',
        'como uso o interfone de emergencia',
        'estou me sentindo em risco',
        'preciso comunicar uma ocorrencia de seguranca'
    ]
}

if USAR_CSV_PROPRIO:
    from google.colab import files
    print(f'Envie {NOME_CSV} (ou outro CSV com texto e intencao).')
    enviados = files.upload()
    caminho = next(iter(enviados))
    df = pd.read_csv(caminho)
else:
    registros = [
        {'texto': texto, 'intencao': intencao}
        for intencao, textos in dados_demo.items()
        for texto in textos
    ]
    df = pd.DataFrame(registros)

colunas_obrigatorias = {'texto', 'intencao'}
if not colunas_obrigatorias.issubset(df.columns):
    raise ValueError(f'O CSV precisa conter as colunas {colunas_obrigatorias}.')

df = df[['texto', 'intencao']].dropna().drop_duplicates().reset_index(drop=True)
df['texto'] = df['texto'].astype(str).str.strip()
df['intencao'] = df['intencao'].astype(str).str.strip()
df = df[(df['texto'] != '') & (df['intencao'] != '')].reset_index(drop=True)

contagens = df['intencao'].value_counts()
if (contagens < 5).any():
    raise ValueError('Cada intenção precisa de pelo menos 5 exemplos para a divisão estratificada.')

print(f'{len(df)} mensagens | {df.intencao.nunique()} intenções')
display(df.sample(8, random_state=SEED))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
ordem = df['intencao'].value_counts().index
sns.countplot(data=df, y='intencao', order=ordem, ax=axes[0])
axes[0].set_title('Distribuição das intenções')
axes[0].set_xlabel('Quantidade')
axes[0].set_ylabel('Intenção')

df['tamanho_original'] = df['texto'].str.split().str.len()
sns.histplot(df['tamanho_original'], bins=15, kde=True, ax=axes[1])
axes[1].set_title('Comprimento das mensagens')
axes[1].set_xlabel('Número de palavras')
plt.tight_layout()
plt.show()
display(df.groupby('intencao')['tamanho_original'].agg(['count', 'mean', 'median', 'max']).round(2))


## 2. Pré-processamento

A normalização converte para minúsculas, remove URLs, e-mails, acentos, pontuação e espaços duplicados. Números são substituídos por `NUM`, preservando a informação de que havia um valor.

Não removemos *stopwords* indiscriminadamente: em mensagens curtas, termos como **não**, **como** e **onde** carregam intenção. O mesmo pré-processamento será reutilizado na inferência para evitar divergência entre treino e produção.

In [ ]:
def normalizar_texto(texto):
    texto = str(texto).lower().strip()
    texto = re.sub(r'https?://\S+|www\.\S+', ' URL ', texto)
    texto = re.sub(r'\S+@\S+', ' EMAIL ', texto)
    texto = unidecode(texto)
    texto = re.sub(r'\d+(?:[.,]\d+)?', ' NUM ', texto)
    texto = re.sub(r'[^a-zA-Z_\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def tokenizar(texto):
    return normalizar_texto(texto).split()

df['texto_limpo'] = df['texto'].map(normalizar_texto)
df['tokens'] = df['texto'].map(tokenizar)
df = df[df['texto_limpo'].str.len() > 0].reset_index(drop=True)

display(df[['texto', 'texto_limpo', 'tokens']].sample(6, random_state=SEED))
print('Vocabulário total (apenas diagnóstico):', len({t for tokens in df.tokens for t in tokens}))


## 3. Separação sem vazamento de dados

A divisão é estratificada em **70% treino, 15% validação e 15% teste**. O teste fica isolado até a avaliação final. O Word2Vec, o tokenizador e o encoder são ajustados depois da divisão.

In [ ]:
treino, temp = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df['intencao']
)
validacao, teste = train_test_split(
    temp, test_size=0.50, random_state=SEED, stratify=temp['intencao']
)

treino = treino.reset_index(drop=True)
validacao = validacao.reset_index(drop=True)
teste = teste.reset_index(drop=True)

print(f'Treino: {len(treino)} | Validação: {len(validacao)} | Teste: {len(teste)}')
display(pd.concat({
    'treino': treino.intencao.value_counts(),
    'validacao': validacao.intencao.value_counts(),
    'teste': teste.intencao.value_counts()
}, axis=1).fillna(0).astype(int))


## 4. Word2Vec

O **Word2Vec** aprende vetores densos a partir do contexto das palavras. Usaremos Skip-gram (`sg=1`), apropriado para corpus pequeno.

- `vector_size=100`: cada palavra vira um vetor de 100 dimensões;
- `window=4`: contexto de até quatro posições;
- `min_count=1`: mantém palavras raras na base didática;
- treinamento feito **somente nos textos de treino**.

Em uma base de produção, prefira mais dados e valide `min_count`, dimensão e janela por experimento.

In [ ]:
DIM_EMBEDDING = 100

w2v = Word2Vec(
    sentences=treino['tokens'].tolist(),
    vector_size=DIM_EMBEDDING,
    window=4,
    min_count=1,
    workers=1,
    sg=1,
    negative=10,
    epochs=150,
    seed=SEED
)

print('Palavras no Word2Vec:', len(w2v.wv))
for palavra in ['metro', 'bicicleta', 'elevador', 'bilhete']:
    if palavra in w2v.wv:
        print(f'\nMais próximas de {palavra!r}:')
        print(w2v.wv.most_similar(palavra, topn=5))


In [ ]:
# Projeção 2D de palavras frequentes — apenas para interpretação visual
palavras_plot = [p for p in w2v.wv.index_to_key[:40] if p not in {'a', 'o', 'de', 'do', 'da', 'e'}]
vetores_plot = np.vstack([w2v.wv[p] for p in palavras_plot])
coords = PCA(n_components=2, random_state=SEED).fit_transform(vetores_plot)

plt.figure(figsize=(13, 8))
plt.scatter(coords[:, 0], coords[:, 1], alpha=0.65)
for palavra, (x, y) in zip(palavras_plot, coords):
    plt.annotate(palavra, (x, y), fontsize=9, alpha=0.85)
plt.title('Projeção PCA dos embeddings Word2Vec')
plt.xlabel('Componente principal 1')
plt.ylabel('Componente principal 2')
plt.show()


## 5. Baseline: média dos embeddings + Regressão Logística

Uma mensagem possui várias palavras, mas o classificador clássico espera um único vetor. A baseline calcula a média dos vetores Word2Vec conhecidos. Ela é rápida e oferece uma referência para avaliar se a rede neural realmente agrega valor.

In [ ]:
def vetor_medio(tokens, modelo_w2v):
    vetores = [modelo_w2v.wv[t] for t in tokens if t in modelo_w2v.wv]
    if not vetores:
        return np.zeros(modelo_w2v.vector_size, dtype=np.float32)
    return np.mean(vetores, axis=0)

X_train_w2v = np.vstack(treino.tokens.map(lambda x: vetor_medio(x, w2v)))
X_val_w2v = np.vstack(validacao.tokens.map(lambda x: vetor_medio(x, w2v)))
X_test_w2v = np.vstack(teste.tokens.map(lambda x: vetor_medio(x, w2v)))

baseline = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED)
baseline.fit(X_train_w2v, treino.intencao)
pred_base = baseline.predict(X_test_w2v)

acc_base = accuracy_score(teste.intencao, pred_base)
f1_base = f1_score(teste.intencao, pred_base, average='macro')
print(f'Baseline — acurácia: {acc_base:.3f} | F1 macro: {f1_base:.3f}')
print(classification_report(teste.intencao, pred_base, zero_division=0))


## 6. Sequências e matriz de embeddings

**Word2Vec** é o algoritmo que aprende representações de palavras. A camada **Embedding** é o componente da rede que transforma índices inteiros nesses vetores durante o treinamento. Aqui, a matriz da camada é inicializada com os vetores Word2Vec; palavras fora do Word2Vec recebem pequenos valores aleatórios.

O tokenizador é ajustado exclusivamente no treino. O tamanho máximo da sequência usa o percentil 95 dos comprimentos do treino, evitando que poucas mensagens longas desperdicem memória.

In [ ]:
tokenizer = Tokenizer(oov_token='<OOV>', filters='')
tokenizer.fit_on_texts(treino['texto_limpo'])

MAX_LEN = max(5, int(np.percentile(treino.tokens.str.len(), 95)))
VOCAB_SIZE = len(tokenizer.word_index) + 1

def preparar_sequencias(series):
    seq = tokenizer.texts_to_sequences(series)
    return pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')

X_train = preparar_sequencias(treino.texto_limpo)
X_val = preparar_sequencias(validacao.texto_limpo)
X_test = preparar_sequencias(teste.texto_limpo)

encoder = LabelEncoder()
y_train = encoder.fit_transform(treino.intencao)
y_val = encoder.transform(validacao.intencao)
y_test = encoder.transform(teste.intencao)
N_CLASSES = len(encoder.classes_)

rng = np.random.default_rng(SEED)
embedding_matrix = rng.normal(0, 0.05, (VOCAB_SIZE, DIM_EMBEDDING)).astype(np.float32)
embedding_matrix[0] = 0
cobertas = 0
for palavra, idx in tokenizer.word_index.items():
    if palavra in w2v.wv:
        embedding_matrix[idx] = w2v.wv[palavra]
        cobertas += 1

print('Vocabulário da rede:', VOCAB_SIZE)
print('Tamanho máximo:', MAX_LEN)
print(f'Cobertura inicial do Word2Vec: {cobertas}/{VOCAB_SIZE - 1} ({cobertas/(VOCAB_SIZE-1):.1%})')
print('Classes:', list(encoder.classes_))


## 7. Modelo neural BiLSTM

A BiLSTM lê a frase nas duas direções e captura dependências de ordem, algo que a média de vetores da baseline não faz. A camada de embeddings começa com Word2Vec e permanece treinável para se adaptar à classificação de intenções. *Dropout*, redução da taxa de aprendizado e parada antecipada ajudam a reduzir sobreajuste.

In [ ]:
tf.keras.backend.clear_session()
modelo = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=DIM_EMBEDDING,
        weights=[embedding_matrix],
        trainable=True,
        mask_zero=True,
        name='embedding_word2vec'
    ),
    SpatialDropout1D(0.25),
    Bidirectional(LSTM(64, return_sequences=True, dropout=0.20)),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.35),
    Dense(N_CLASSES, activation='softmax')
])

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
modelo.build(input_shape=(None, MAX_LEN))
modelo.summary()


In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, min_delta=1e-4),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1)
]

historico = modelo.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
hist = pd.DataFrame(historico.history)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
hist[['loss', 'val_loss']].plot(ax=axes[0], title='Perda por época')
hist[['accuracy', 'val_accuracy']].plot(ax=axes[1], title='Acurácia por época')
for ax in axes:
    ax.set_xlabel('Época')
plt.tight_layout()
plt.show()


## 8. Avaliação final

Acurácia mede o total de acertos. **F1 macro** calcula o F1 de cada intenção e dá o mesmo peso a todas as classes, por isso é importante quando queremos qualidade equilibrada. A matriz de confusão mostra quais intenções o modelo troca entre si.

> Como a base de demonstração é pequena, os valores variam e não devem ser tratados como evidência de prontidão para produção.

In [ ]:
prob_teste = modelo.predict(X_test, verbose=0)
pred_idx = prob_teste.argmax(axis=1)
pred_neural = encoder.inverse_transform(pred_idx)

acc_neural = accuracy_score(teste.intencao, pred_neural)
f1_neural = f1_score(teste.intencao, pred_neural, average='macro')
print(f'BiLSTM — acurácia: {acc_neural:.3f} | F1 macro: {f1_neural:.3f}\n')
print(classification_report(
    teste.intencao, pred_neural, labels=encoder.classes_, zero_division=0
))

cm = confusion_matrix(teste.intencao, pred_neural, labels=encoder.classes_)
plt.figure(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title('Matriz de confusão — conjunto de teste')
plt.xlabel('Predição')
plt.ylabel('Classe real')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

comparacao = pd.DataFrame({
    'modelo': ['Média Word2Vec + Regressão Logística', 'Word2Vec + BiLSTM'],
    'acuracia_teste': [acc_base, acc_neural],
    'f1_macro_teste': [f1_base, f1_neural]
}).sort_values('f1_macro_teste', ascending=False)
display(comparacao.style.format({'acuracia_teste': '{:.3f}', 'f1_macro_teste': '{:.3f}'}))


In [ ]:
# Inspeção dos erros com a confiança atribuída
analise_erros = teste[['texto', 'intencao']].copy()
analise_erros['predicao'] = pred_neural
analise_erros['confianca'] = prob_teste.max(axis=1)
analise_erros = analise_erros[analise_erros.intencao != analise_erros.predicao]
display(analise_erros.sort_values('confianca', ascending=False))


## 9. Inferência

A função abaixo aplica exatamente a mesma normalização e retorna as três intenções mais prováveis. Se a confiança máxima ficar abaixo do limiar, o sistema recomenda encaminhamento humano em vez de fingir certeza. O limiar deve ser calibrado com dados reais e conforme o custo de cada erro.

In [ ]:
RESPOSTAS = {
    'acessibilidade': 'Vou orientar uma rota acessível e o apoio da equipe.',
    'achados_perdidos': 'Vou orientar o registro no canal de achados e perdidos.',
    'bilhetes_tarifas': 'Vou ajudar com bilhetes, recarga ou tarifa.',
    'bicicletas': 'Vou apresentar as regras e horários para transportar bicicleta.',
    'reclamacoes_sugestoes': 'Vou orientar o canal de atendimento e protocolo.',
    'equipamentos': 'Vou verificar a alternativa disponível e o registro de manutenção.',
    'status_operacao': 'Vou consultar o estado atual da operação.',
    'horarios': 'Vou consultar os horários de funcionamento.',
    'rotas_estacoes': 'Vou ajudar a planejar a rota e as integrações.',
    'seguranca_emergencia': 'Situação de segurança: procure ou acione imediatamente um agente no local.'
}

def classificar_intencao(texto, limiar=0.55, top_k=3):
    limpo = normalizar_texto(texto)
    seq = preparar_sequencias(pd.Series([limpo]))
    probs = modelo.predict(seq, verbose=0)[0]
    ordem = np.argsort(probs)[::-1][:top_k]
    top = [
        {'intencao': str(encoder.classes_[i]), 'probabilidade': round(float(probs[i]), 4)}
        for i in ordem
    ]
    melhor = top[0]
    baixa_confianca = melhor['probabilidade'] < limiar
    return {
        'texto_original': texto,
        'texto_normalizado': limpo,
        'intencao': 'encaminhar_atendente' if baixa_confianca else melhor['intencao'],
        'confianca': melhor['probabilidade'],
        'baixa_confianca': baixa_confianca,
        'resposta_sugerida': (
            'Não tenho confiança suficiente; vou encaminhar para atendimento humano.'
            if baixa_confianca else RESPOSTAS[melhor['intencao']]
        ),
        'top_intencoes': top
    }

exemplos = [
    'Esqueci uma bolsa no trem ontem à noite',
    'Posso entrar com a minha bike no sábado?',
    'A escada rolante da estação está parada',
    'Tem alguém passando mal aqui no vagão',
    'Quero saber uma coisa completamente fora do domínio'
]
for exemplo in exemplos:
    print(json.dumps(classificar_intencao(exemplo), ensure_ascii=False, indent=2), '\n')


## 10. Exportação dos artefatos

O pacote exportado contém o modelo neural, o Word2Vec, o tokenizador, o encoder, as configurações e as métricas. Em produção, versione também a base e o código de pré-processamento; o arquivo de configuração registra os parâmetros necessários para reconstruir a entrada.

In [ ]:
PASTA_ARTEFATOS = Path('artefatos_modelo_pln')
PASTA_ARTEFATOS.mkdir(exist_ok=True)

modelo.save(PASTA_ARTEFATOS / 'modelo_intencoes.keras')
w2v.save(str(PASTA_ARTEFATOS / 'word2vec.model'))
joblib.dump(tokenizer, PASTA_ARTEFATOS / 'tokenizer.joblib')
joblib.dump(encoder, PASTA_ARTEFATOS / 'label_encoder.joblib')
joblib.dump(baseline, PASTA_ARTEFATOS / 'baseline_logistica.joblib')

config = {
    'max_len': MAX_LEN,
    'dim_embedding': DIM_EMBEDDING,
    'vocab_size': VOCAB_SIZE,
    'classes': encoder.classes_.tolist(),
    'limiar_recomendado_inicial': 0.55,
    'seed': SEED,
    'preprocessamento': 'minusculas, URL/EMAIL, unidecode, NUM, remocao de pontuacao'
}
metricas = {
    'baseline': {'acuracia': float(acc_base), 'f1_macro': float(f1_base)},
    'bilstm': {'acuracia': float(acc_neural), 'f1_macro': float(f1_neural)},
}
(PASTA_ARTEFATOS / 'config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
(PASTA_ARTEFATOS / 'metricas.json').write_text(json.dumps(metricas, ensure_ascii=False, indent=2), encoding='utf-8')
df[['texto', 'intencao']].to_csv(PASTA_ARTEFATOS / 'dados_utilizados.csv', index=False)

arquivo_zip = shutil.make_archive('artefatos_modelo_pln', 'zip', PASTA_ARTEFATOS)
print('Pacote criado:', arquivo_zip)
print('Arquivos:', [p.name for p in PASTA_ARTEFATOS.iterdir()])

# Descomente no Google Colab para baixar o pacote:
# from google.colab import files
# files.download(arquivo_zip)


## Conclusões e próximos passos

O pipeline completo foi construído: dados → limpeza → divisão → Word2Vec → baseline → BiLSTM com embedding → avaliação → inferência → exportação.

Antes de uso real:

1. coletar e anonimizar mensagens reais conforme LGPD;
2. criar um guia de rotulagem e medir concordância entre anotadores;
3. aumentar a base, especialmente nas intenções confundidas;
4. deduplicar mensagens antes da divisão e usar validação cruzada;
5. calibrar o limiar de confiança e medir precisão/cobertura;
6. monitorar mudança de vocabulário, classes e desempenho;
7. comparar com TF-IDF, FastText e um Transformer em português;
8. manter emergências com regras e encaminhamento humano, sem depender apenas do classificador.

**Importante:** a classificação identifica a intenção, mas não consulta dados operacionais em tempo real. Respostas sobre horários, tarifas e status devem vir de fontes oficiais atualizadas.